In [1]:
# Pickle, Save & Load Model
import os
import pickle

def save_model(model, filename):
    with open(filename, 'wb') as file:
        pickle.dump(model, file)
        
def load_model(filename):
    if os.path.exists('filename'):
        with open(filename, 'rb') as file:
            return pickle.load(file)
    else:
        return None

In [2]:
# Read CSV
import pandas as pd

import kagglehub
from pathlib import Path

# Massive Rotten Tomatoes Movies and Reviews, downloaded once to the kagglehub cache
RT_DIR = Path(kagglehub.dataset_download("andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews"))
movies = pd.read_csv(RT_DIR / "rotten_tomatoes_movies.csv", usecols=["id", "title", "audienceScore"]).dropna()
reviews = pd.read_csv(RT_DIR / "rotten_tomatoes_movie_reviews.csv", usecols=["id", "reviewText", "scoreSentiment"]).dropna()

dataset = (
    reviews.merge(movies, on="id")
    .rename(columns={"reviewText": "review", "scoreSentiment": "sentimentScore"})[["title", "review", "sentimentScore"]]
    .sample(1000)
)
dataset = dataset.dropna(subset="review")
dataset = dataset.astype(str)
dataset

C:\Users\jorda\Code\ml-notebooks\courses\natural-language-processing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,title,review,sentimentScore
171006,Pianomania,"Ninety-three minutes pass like a bar of the ""M...",POSITIVE
1167719,See Spot Run,"Derivative, even desperately goofy in its ques...",POSITIVE
28463,The Blind Side,"The Blind Side fumbles a remarkable, true stor...",NEGATIVE
970509,Training Day,"Unquestionably convincing, disturbingly realis...",POSITIVE
1152731,The Story of Us,The entire film is a haphazard scrapbook of hi...,NEGATIVE
...,...,...,...
60795,Finding Fela,"...even in scenes of Kuti in his prime, Findin...",NEGATIVE
667747,Bacurau,The tone of the movie so unexpected but deligh...,POSITIVE
679992,Pooh's Heffalump Movie,...enfolds young viewers like a favorite blanket.,POSITIVE
99275,Seeking a Friend for the End of the World,The mash-up of skewed love and black humour sh...,POSITIVE


In [3]:
review_list = dataset['review'].to_list()
label_list = dataset['sentimentScore'].to_list()

In [4]:
# Preprocessing
# Jadiin semua huruf kecil
# Remove punctuation
# Remove stopwords
# Stemming, lemmatization + pos_tag

import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag

In [5]:
def get_tag(tag):
    if tag == 'jj':
        return 'a'
    elif tag in ['nn', 'vb', 'rb']:
        return tag[0]
    else:
        return 'n'

In [6]:
def preprocess(review_list):
    punctuation = string.punctuation
    stop_words = stopwords.words('english')
    stemmer = SnowballStemmer('english')
    lemmatizer = WordNetLemmatizer()
    
    preprocessed_list = []
    
    for review in review_list:
        review = review.lower()                                                  # Huruf kecil
        review = ''.join([char for char in review if char not in punctuation])   # Hapus punctuation
        
        tokens =  word_tokenize(review)
        tokens = [token for token in tokens if token not in stop_words]          # Hapus stopwords
        
        stemmed_tokens = [stemmer.stem(token) for token in tokens]               # Stemming
        
        tagged_tokens = pos_tag(stemmed_tokens)                                  # Lemmatization, perlu tag
        lemmatized_tokens = []
        for token, tag in tagged_tokens:
            lemmatized_token = lemmatizer.lemmatize(token, get_tag(tag.lower()))
            lemmatized_tokens.append(lemmatized_token)
        
        preprocessed_review = ' '.join(lemmatized_tokens)
        preprocessed_list.append(preprocessed_review)
    
    return preprocessed_list  

In [7]:
preprocess(review_list)

['ninetythre minut pas like bar minut waltz',
 'deriv even desper goofi quest leav gag potenti unreal youll also laugh much probabl wont care',
 'blind side fumbl remark true stori africanamerican product west memphi project end christian school care wealthi white famili went profession footbal glori',
 'unquestion convinc disturb realist unflinch honest',
 'entir film haphazard scrapbook high low',
 'princebythewood isnt reinvent wheel succeed pure entertain level',
 'scari stori documentari isnt book creator fan foe particular schwartz son peter grand person arc leav u satisfi sen redempt',
 'gloomi provid mani summerblockbust thrill silli serious arthous allegori film best left completist isnt profound move film although sometim ape one',
 'mcqueen make case subject artist whose clay cloth also despit give short shrift psychoanalysi remind u everyth might want know artist found art',
 'misogynist homophob miser paraphras lui buuel extermin angel good way full review spanish',
 'film

In [8]:
from pathlib import Path

MODEL_DIR = Path.home() / ".cache" / "ml-notebooks" / "natural-language-processing/quiz-answer-1-movie-sentiment"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
# Train model
from nltk.classify import NaiveBayesClassifier, accuracy
from nltk.probability import FreqDist
import random
# Semua kata di satu list
# Tentuin yang mana jadi fitur
# extract_features
# Train mode, jangan lupa disave

def train_model():
    reviews = preprocess(review_list)
    all_words = []
    
    for review in reviews:
        all_words.extend(word_tokenize(review))
    
    fd = FreqDist(all_words)
    
    word_features = [word for word, _ in fd.most_common(500)]
    
    def extract_features(review):
        tokens = word_tokenize(review)
        return {word : (word in tokens) for word in word_features}
    
    feature_set = [(extract_features(review), label) for review, label in zip(reviews, label_list)]
    
    random.shuffle(feature_set) # Agar dia random dan tidak bias
    train_length = int(len(feature_set) * 0.8)
    
    train_set = feature_set[:train_length]
    test_set = feature_set[train_length:]
    
    classifier = NaiveBayesClassifier.train(train_set)
    model_accuracy = accuracy(classifier, test_set)
    
    print("Model Accuracy: ", model_accuracy)
    save_model(classifier, MODEL_DIR / "model.pickle")
    return classifier

In [9]:
train_model()

Model Accuracy:  0.735


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(tokenizer=word_tokenize, stop_words='english', ngram_range=(1, 3))
all_words_vector = vectorizer.fit_transform(review_list)

def get_recommendation(review):
    review_vector = vectorizer.transform([review])
    similarity = cosine_similarity(review_vector, all_words_vector).flatten()
    top = similarity.argsort()[-2:]
    
    print("Top movie title: ")
    for idx in top:
        print(dataset['title'].iloc[idx])

C:\Users\jorda\Code\ml-notebooks\courses\natural-language-processing\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [11]:
get_recommendation("This movie is the best I've ever seen")

Top movie title: 
Hunt for the Wilderpeople
A Cold Wind in August


In [12]:
import spacy

def extract_ner(review):
    nlp = spacy.load('en_core_web_sm')
    # python -m spacy download en_core_web_sm <-
    
    doc = nlp(review)
    named_entities = {
        'language' : [ent.text for ent in doc.ents if ent.label_ == "LANGUAGE"],
        'location' : [ent.text for ent in doc.ents if ent.label_ in ['GPE', 'LOC']]
    }
    
    return named_entities

In [13]:
ner = extract_ner("This English movie is the best I've ever seen shot in Nile River")
print(ner['language'])
print(ner['location'])

[]
['Nile River']
